### Первая версия(самая простая)

In [ ]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from xgboost import XGBClassifier
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from scipy.sparse import hstack
import time

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

In [ ]:
def prepare_data(df):
    df["text"] = df["title"].fillna("") + " " + df["body"].fillna("")

    df["clean_text"] = df["text"].apply(preprocess_text)
    return df

In [ ]:
def get_vectorizer(method="tfidf"):
    if method == "tfidf":
        return TfidfVectorizer(
            max_features=15000,
            ngram_range=(1,2),   
            min_df=3,
            max_df=0.9
        )

    elif method == "bow":
        return CountVectorizer(
            max_features=10000,
            ngram_range=(1,2),
            min_df=3
        )

In [ ]:
def train_word2vec(texts):
    sentences = [t.split() for t in texts]

    model = Word2Vec(
        sentences,
        vector_size=100,
        window=5,
        min_count=2
    )
    return model


def vectorize_w2v(texts, model, tfidf=None):
    vectors = []

    for text in texts:
        words = text.split()
        word_vecs = []
        weights = []

        for w in words:
            if w in model.wv:
                weight = tfidf.get(w, 1.0) if tfidf else 1.0
                word_vecs.append(model.wv[w] * weight)
                weights.append(weight)

        if len(word_vecs) == 0:
            vectors.append(np.zeros(model.vector_size))
        else:
            vectors.append(np.sum(word_vecs, axis=0) / np.sum(weights))

    return np.array(vectors)

In [ ]:
def get_model(name="logreg", C=1.0):

    if name == "logreg":
        return LogisticRegression(
            max_iter=4000,
            C=C,
            solver="liblinear",
            class_weight='balanced'
        )

    elif name == "svm":
        return LinearSVC(
            C=C,
            max_iter=7000,
            class_weight='balanced'
        )

    elif name == "nb":
        return MultinomialNB(alpha=0.5)

    else:
        raise ValueError("Unknown model")

In [ ]:
def run_experiment(
    df,
    text_col,
    target_col,
    vectorizer_type="tfidf",
    model_name="logreg",
    C=1.0
):

    X = df[text_col]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    vectorizer = get_vectorizer(vectorizer_type)

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec = vectorizer.transform(X_test)

    model = get_model(model_name, C=C)
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)

    f1 = f1_score(y_test, preds)

    return f1

In [ ]:
def run_all_experiments(df):

    configs = [
        ("tfidf", "svm", 0.5),
        ("tfidf", "svm", 1.0),
        ("tfidf", "svm", 1.5),
        ("tfidf", "svm", 2.0),

        ("tfidf", "logreg", 0.5),
        ("tfidf", "logreg", 1.0),
        ("tfidf", "logreg", 2.0),
    ]

    results = []

    for vec, model, C in configs:
        f1 = run_experiment(
            df,
            text_col="clean_text",
            target_col="label",
            vectorizer_type=vec,
            model_name=model,
            C=C   
        )

        results.append((vec, model, C, f1))

    return results

In [ ]:
df = pd.read_csv("/content/train.csv")
df = prepare_data(df)

In [ ]:
results = run_all_experiments(df)

for r in results:
    print(f"Vectorizer: {r[0]}, Model: {r[1]}, C: {r[2]}, F1: {r[3]:.4f}")

Vectorizer: tfidf, Model: svm, C: 0.5, F1: 0.7967
Vectorizer: tfidf, Model: svm, C: 1.0, F1: 0.7887
Vectorizer: tfidf, Model: svm, C: 1.5, F1: 0.7749
Vectorizer: tfidf, Model: svm, C: 2.0, F1: 0.7760
Vectorizer: tfidf, Model: logreg, C: 0.5, F1: 0.7711
Vectorizer: tfidf, Model: logreg, C: 1.0, F1: 0.7834
Vectorizer: tfidf, Model: logreg, C: 2.0, F1: 0.7995

### 2 версия 0,82

In [ ]:
def prepare_data(df):
    df["title"] = df["title"].fillna("").astype(str)
    df["body"]  = df["body"].fillna("").astype(str)
    df["text"]  = df["title"] + " " + df["body"]

    df["clean_title"] = df["title"].apply(preprocess_text)
    df["clean_body"]  = df["body"].apply(preprocess_text)
    df["clean_text"]  = df["clean_title"] + " " + df["clean_body"]

    df["title_len"]        = df["title"].str.len()
    df["body_len"]         = df["body"].str.len()
    df["word_count"]       = df["text"].str.split().str.len()
    df["title_word_count"] = df["title"].str.split().str.len()
    df["body_word_count"]  = df["body"].str.split().str.len()
    df["title_body_ratio"] = df["title_len"] / (df["body_len"] + 1)

    depression_keywords = [
        'depress', 'depressed', 'depression', 'sad', 'lonely', 'alone',
        'hopeless', 'suicide', 'suicidal', 'anxiety', 'anxious', 'kill',
        'die', 'death', 'hurt', 'pain', 'cry', 'tears', 'empty', 'worthless'
    ]
    for kw in depression_keywords:
        df[f'kw_{kw}'] = df['text'].str.lower().str.count(kw)

    return df

In [ ]:
def clean_text_light(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[^a-z\s\.,!?\'"]', '', text)
    return text.strip()

In [ ]:
def run_hybrid_stacking(df):
    start_time = time.time()

    df["text_light"] = (df["title"] + " " + df["body"]).apply(clean_text_light)

    X = df["text_light"]
    y = df["label"]

    exclude = {'id', 'title', 'body', 'text', 'clean_title', 'clean_body', 'clean_text', 'label', 'text_light'}
    meta_cols = [col for col in df.columns if col not in exclude and df[col].dtype in ['int64', 'float64']]
    X_meta = df[meta_cols].values.astype(float)

    X_train, X_test, y_train, y_test, X_meta_train, X_meta_test = train_test_split(
        X, y, X_meta, test_size=0.2, random_state=42, stratify=y
    )

    print("   1. TF-IDF (word + char)...")
    word_vec = TfidfVectorizer(ngram_range=(1, 3), max_features=15000, min_df=2, sublinear_tf=True)
    char_vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=8000, min_df=3, sublinear_tf=True)
    word_vec.fit(X_train)
    char_vec.fit(X_train)

    X_train_word = word_vec.transform(X_train)
    X_test_word  = word_vec.transform(X_test)
    X_train_char = char_vec.transform(X_train)
    X_test_char  = char_vec.transform(X_test)

    print("   2. Doc2Vec (20 эпох)...")
    tagged_train = [TaggedDocument(words=doc.split(), tags=[str(i)]) for i, doc in enumerate(X_train)]
    d2v_model = Doc2Vec(vector_size=200, window=5, min_count=2, workers=4,
                        epochs=20, dm=0, seed=42)
    d2v_model.build_vocab(tagged_train)
    d2v_model.train(tagged_train, total_examples=d2v_model.corpus_count, epochs=d2v_model.epochs)

    class Doc2VecTransformer:
        def __init__(self, model): self.model = model
        def transform(self, texts):
            return np.array([self.model.infer_vector(t.split()) for t in texts])
        def fit(self, X, y=None): return self

    d2v_trans = Doc2VecTransformer(d2v_model)
    X_train_d2v = d2v_trans.transform(X_train)
    X_test_d2v  = d2v_trans.transform(X_test)

    X_train_full = hstack([X_train_word, X_train_char, X_train_d2v, X_meta_train])
    X_test_full  = hstack([X_test_word,  X_test_char,  X_test_d2v,  X_meta_test])

    estimators = [
        ('lr',  LogisticRegression(C=10, max_iter=3000, class_weight='balanced', random_state=42)),
        ('rf',  RandomForestClassifier(n_estimators=250, max_depth=12, n_jobs=-1, random_state=42, class_weight='balanced')),
        ('xgb', XGBClassifier(n_estimators=300, learning_rate=0.06, max_depth=7,
                              scale_pos_weight=y_train.value_counts()[0]/y_train.value_counts()[1],
                              subsample=0.85, colsample_bytree=0.85, random_state=42, n_jobs=-1))
    ]

    stacking = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=5000, class_weight='balanced', random_state=42),
        cv=3,
        n_jobs=-1,
        passthrough=True
    )

    stacking.fit(X_train_full, y_train)
    preds = stacking.predict(X_test_full)
    f1 = f1_score(y_test, preds)

    elapsed = time.time() - start_time
    print(f"\n F1 = {f1:.4f}")
    print(f"Время: {elapsed//60:.0f} мин {elapsed%60:.0f} сек")
    return f1

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    tokens = [lemmatizer.lemmatize(w) for w in tokens]

    return " ".join(tokens)

In [ ]:
df = pd.read_csv("/content/train.csv")
df = prepare_data(df)   
hybrid_f1 = run_hybrid_stacking(df)